# Bài 03: Đánh giá Tổng quan với Đường cong ROC và Chỉ số AUC

**Mục tiêu bài học:**
- Hiểu hai khái niệm mới: Tỷ lệ True Positive (TPR) và Tỷ lệ False Positive (FPR).
- Nắm được cách xây dựng và ý nghĩa của đường cong ROC (Receiver Operating Characteristic).
- Hiểu được chỉ số AUC (Area Under the Curve) và tại sao nó là một thước đo tổng quan, không phụ thuộc vào ngưỡng quyết định.
- Biết khi nào nên dùng ROC/AUC và khi nào nên dùng Precision-Recall Curve.

## Vấn đề của Precision-Recall và F1-Score

Ở bài trước, chúng ta đã thấy Precision, Recall, và F1-Score rất hữu ích, nhưng chúng có một đặc điểm chung: giá trị của chúng **phụ thuộc vào ngưỡng quyết định (decision threshold)** mà chúng ta chọn.

Điều này đặt ra một câu hỏi: Liệu có cách nào để đánh giá "sức mạnh" tổng thể của một mô hình, độc lập với bất kỳ ngưỡng cụ thể nào không? Chúng ta muốn biết, nhìn chung, mô hình này có khả năng phân biệt giữa lớp `Positive` và `Negative` tốt đến mức nào.

Đây là lúc đường cong **ROC** và chỉ số **AUC** tỏa sáng.

## Hai khái niệm mới: TPR và FPR

Để hiểu đường cong ROC, chúng ta cần làm quen với hai tỷ lệ được tính từ Confusion Matrix:

1.  **True Positive Rate (TPR):** Tỷ lệ các trường hợp `Positive` thật sự đã được dự đoán đúng là `Positive`.
    $$ \text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}} $$
    Bạn có nhận ra công thức này không? Nó **chính là Recall** (hay còn gọi là Sensitivity). TPR trả lời câu hỏi: "Mô hình đã 'vớt' được bao nhiêu phần trăm trong tổng số các trường hợp Positive?"

2.  **False Positive Rate (FPR):** Tỷ lệ các trường hợp `Negative` thật sự đã bị dự đoán sai là `Positive`.
    $$ \text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}} $$
    FPR trả lời câu hỏi: "Mô hình đã báo động nhầm (dự đoán là Positive) trên bao nhiêu phần trăm trong tổng số các trường hợp Negative?"

Một mô hình lý tưởng sẽ có **TPR cao** (tìm thấy tất cả các trường hợp Positive) và **FPR thấp** (không báo động nhầm).

## Đường cong ROC (Receiver Operating Characteristic)

Đường cong ROC là một biểu đồ thể hiện hiệu năng của một mô hình phân loại ở tất cả các ngưỡng quyết định. Nó vẽ mối quan hệ giữa **TPR** và **FPR**.

- **Trục tung:** True Positive Rate (TPR) - tức là Recall.
- **Trục hoành:** False Positive Rate (FPR).

Giống như Precision-Recall Curve, mỗi điểm trên đường cong ROC tương ứng với một ngưỡng quyết định. Khi chúng ta thay đổi ngưỡng từ 0 đến 1, chúng ta sẽ vẽ ra được đường cong này.

**Cách đọc một đường cong ROC:**

- **Góc trên bên trái (TPR=1, FPR=0):** Đây là điểm của một mô hình hoàn hảo. Nó phân loại đúng tất cả các trường hợp Positive (TPR=1) và không phân loại sai bất kỳ trường hợp Negative nào (FPR=0).
- **Đường chéo (y=x):** Đây là đường của một mô hình đoán ngẫu nhiên. Ví dụ, một mô hình tung đồng xu để dự đoán. Đường ROC của nó sẽ nằm trên đường chéo này. Bất kỳ mô hình nào có đường cong nằm dưới đường này đều được coi là tệ hơn cả đoán ngẫu nhiên.
- **Càng gần góc trên bên trái càng tốt:** Một mô hình có đường cong ROC càng "phình" về phía góc trên bên trái thì khả năng phân tách hai lớp của nó càng tốt.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt

# Sử dụng lại dữ liệu và mô hình từ bài trước
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2, 
                           n_redundant=10, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_scores = model.predict_proba(X_test)[:, 1]

# Tính FPR, TPR và ngưỡng
fpr, tpr, thresholds = roc_curve(y_test, y_scores)

# Vẽ đường cong ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='Logistic Regression')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing') # Đường chéo
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()

## Chỉ số AUC (Area Under the Curve)

Đường cong ROC rất hữu ích để trực quan hóa, nhưng để so sánh các mô hình một cách định lượng, chúng ta cần một con số duy nhất. Con số đó chính là **AUC - Area Under the (ROC) Curve** (Diện tích dưới đường cong ROC).

AUC đo lường toàn bộ diện tích nằm dưới đường cong ROC. Giá trị của AUC nằm trong khoảng từ 0 đến 1.

**Ý nghĩa của AUC:**
- **AUC = 1.0:** Mô hình hoàn hảo. Nó có khả năng phân biệt hoàn toàn giữa lớp Positive và Negative.
- **AUC = 0.5:** Mô hình không có khả năng phân biệt tốt hơn đoán ngẫu nhiên.
- **AUC < 0.5:** Mô hình tệ hơn cả đoán ngẫu nhiên (có thể do gán nhãn dự đoán bị ngược).
- **0.5 < AUC < 1.0:** Mô hình có khả năng phân biệt. Giá trị càng cao, khả năng phân biệt càng tốt.

**Diễn giải AUC theo một cách khác:**
> AUC có thể được hiểu là **xác suất** mà một mẫu **Positive** được chọn ngẫu nhiên sẽ có điểm số (score) cao hơn một mẫu **Negative** được chọn ngẫu nhiên.

Đây là một thước đo rất mạnh mẽ vì nó đánh giá hiệu năng tổng thể của mô hình trên tất cả các ngưỡng, cho chúng ta một cái nhìn tổng quan về khả năng phân tách của mô hình.

In [ ]:
# Tính AUC score
auc_score = roc_auc_score(y_test, y_scores)

print(f"AUC Score: {auc_score:.4f}")

# Hiển thị AUC score trên biểu đồ
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()

Một AUC score là 0.90 cho thấy đây là một mô hình khá tốt trong việc phân biệt giữa hai lớp.

## Khi nào dùng ROC/AUC vs. Precision-Recall Curve?

Cả hai đều là công cụ đánh giá tuyệt vời, nhưng chúng có điểm mạnh riêng và phù hợp với các kịch bản khác nhau.

**Hãy dùng ROC/AUC khi:**
1.  **Dữ liệu tương đối cân bằng:** Khi số lượng mẫu của lớp Positive và Negative gần bằng nhau, ROC/AUC cho một cái nhìn tổng quan tốt.
2.  **Bạn quan tâm đến hiệu năng tổng thể:** Khi bạn muốn so sánh khả năng phân tách vốn có của các mô hình khác nhau, bất kể ngưỡng quyết định sẽ được chọn sau này.
3.  **Chi phí của FP và FN là tương đương:** Khi không có sự chênh lệch lớn về tầm quan trọng giữa việc báo động nhầm (FP) và bỏ sót (FN).

**Hãy dùng Precision-Recall Curve khi:**
1.  **Dữ liệu mất cân bằng (imbalanced):** Đây là thế mạnh lớn nhất của P-R curve. Khi lớp Positive rất hiếm (ví dụ: phát hiện gian lận, chẩn đoán bệnh hiếm), một sự thay đổi nhỏ trong FPR (ví dụ từ 0.01 lên 0.02) có thể không ảnh hưởng nhiều đến ROC curve, nhưng lại có thể làm giảm Precision một cách đáng kể. P-R curve nhạy cảm hơn với sự thay đổi này.
2.  **Bạn quan tâm nhiều hơn đến lớp Positive:** Khi mục tiêu chính của bạn là tìm ra các trường hợp Positive và hiệu quả của các dự đoán Positive đó (ví dụ: tìm kiếm thông tin, gợi ý sản phẩm).
3.  **Chi phí của FP và FN chênh lệch lớn:** Khi bạn cần phải chọn một ngưỡng cụ thể để cân bằng giữa Precision và Recall theo yêu cầu bài toán.

**Quy tắc chung:**
- Bắt đầu với ROC/AUC để có cái nhìn tổng quan và so sánh các mô hình.
- Nếu dữ liệu của bạn mất cân bằng hoặc bạn đặc biệt quan tâm đến hiệu năng trên lớp thiểu số, hãy phân tích sâu hơn bằng Precision-Recall Curve.

## Tóm tắt

- Đường cong **ROC** vẽ mối quan hệ giữa **TPR (Recall)** và **FPR** ở các ngưỡng khác nhau.
- Chỉ số **AUC** là diện tích dưới đường cong ROC, đo lường khả năng phân tách tổng thể của mô hình, không phụ thuộc vào ngưỡng.
- AUC có thể được hiểu là xác suất một mẫu Positive ngẫu nhiên có score cao hơn một mẫu Negative ngẫu nhiên.
- **ROC/AUC** phù hợp cho dữ liệu cân bằng và đánh giá tổng quan.
- **Precision-Recall Curve** phù hợp hơn cho dữ liệu mất cân bằng và khi lớp Positive là trọng tâm.

Đến đây, chúng ta đã có một bộ công cụ khá đầy đủ để đánh giá mô hình. Trong bài học tiếp theo, chúng ta sẽ chính thức đi sâu vào "nhân vật chính" của khóa học: **Vấn đề mất cân bằng dữ liệu (Imbalanced Data)**.